# Data Quality Tests - Bronze Layer
## Schema: bike_store.bronze

This notebook performs comprehensive data quality tests on all bronze layer tables:
- **CRM System**: crm_customer, crm_product, crm_sales
- **ERP System**: erp_customer, erp_category, erp_location

### Test Categories:
1. **Completeness**: Null checks on key fields
2. **Schema Consistency**: Column names and types validation
3. **Data Type Validations**: Proper data types for fields
4. **Range Validations**: Dates, amounts, and numeric ranges
5. **Relationship Validations**: Foreign key integrity
6. **Known Issues**: INT dates and monetary values

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import pandas as pd

# Initialize test results storage
test_results = []

def log_test(table_name, test_category, test_name, status, details=""):
    """Log test result"""
    test_results.append({
        "table": table_name,
        "category": test_category,
        "test": test_name,
        "status": status,
        "details": details,
        "timestamp": datetime.now()
    })
    
def check_completeness(df, table_name, required_columns):
    """Check for null values in required columns"""
    for col in required_columns:
        if col not in df.columns:
            log_test(table_name, "Completeness", f"{col} exists", "FAIL", f"Column {col} not found")
            continue
            
        null_count = df.filter(F.col(col).isNull()).count()
        total_count = df.count()
        null_pct = (null_count / total_count * 100) if total_count > 0 else 0
        
        if null_count > 0:
            log_test(table_name, "Completeness", f"{col} not null", "WARN", 
                    f"{null_count} nulls ({null_pct:.2f}%)")
        else:
            log_test(table_name, "Completeness", f"{col} not null", "PASS", "No nulls found")

def check_schema(df, table_name, expected_schema):
    """Verify schema matches expectations"""
    actual_columns = set(df.columns)
    expected_columns = set(expected_schema.keys())
    
    # Check missing columns
    missing = expected_columns - actual_columns
    if missing:
        log_test(table_name, "Schema", "All columns present", "FAIL", f"Missing: {missing}")
    else:
        log_test(table_name, "Schema", "All columns present", "PASS", "")
    
    # Check extra columns
    extra = actual_columns - expected_columns
    if extra:
        log_test(table_name, "Schema", "No extra columns", "INFO", f"Extra: {extra}")
    
    # Check data types
    for col, expected_type in expected_schema.items():
        if col in df.columns:
            actual_type = dict(df.dtypes)[col]
            if actual_type.lower() != expected_type.lower():
                log_test(table_name, "Schema", f"{col} type", "FAIL", 
                        f"Expected {expected_type}, got {actual_type}")
            else:
                log_test(table_name, "Schema", f"{col} type", "PASS", "")

def check_range(df, table_name, column, min_val=None, max_val=None, allow_null=True):
    """Check value ranges"""
    if column not in df.columns:
        log_test(table_name, "Range", f"{column} range check", "SKIP", "Column not found")
        return
    
    df_filtered = df if allow_null else df.filter(F.col(column).isNotNull())
    
    if min_val is not None:
        invalid_count = df_filtered.filter(F.col(column) < min_val).count()
        if invalid_count > 0:
            log_test(table_name, "Range", f"{column} >= {min_val}", "FAIL", 
                    f"{invalid_count} values below minimum")
        else:
            log_test(table_name, "Range", f"{column} >= {min_val}", "PASS", "")
    
    if max_val is not None:
        invalid_count = df_filtered.filter(F.col(column) > max_val).count()
        if invalid_count > 0:
            log_test(table_name, "Range", f"{column} <= {max_val}", "FAIL", 
                    f"{invalid_count} values above maximum")
        else:
            log_test(table_name, "Range", f"{column} <= {max_val}", "PASS", "")

print("✅ Helper functions loaded")

## 1. bike_store.bronze.crm_customer
Customer data from CRM system

In [0]:
table_name = "bike_store.bronze.crm_customer"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["cst_id", "cst_key", "cst_firstname", "cst_lastname"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "cst_id": "int",
    "cst_key": "string",
    "cst_firstname": "string",
    "cst_lastname": "string",
    "cst_marital_status": "string",
    "cst_gndr": "string",
    "cst_create_date": "date",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Range validations
check_range(df, table_name, "cst_create_date", max_val=datetime.now().date())

# 4. Data quality checks
# Check for duplicate customer IDs
dup_count = df.groupBy("cst_id").count().filter("count > 1").count()
if dup_count > 0:
    log_test(table_name, "Uniqueness", "cst_id unique", "FAIL", f"{dup_count} duplicate IDs")
else:
    log_test(table_name, "Uniqueness", "cst_id unique", "PASS", "")

# Check gender values
valid_genders = df.filter(F.col("cst_gndr").isin(["M", "F", "Male", "Female"])).count()
total = df.filter(F.col("cst_gndr").isNotNull()).count()
if valid_genders < total:
    log_test(table_name, "Validity", "cst_gndr valid values", "WARN", 
            f"{total - valid_genders} invalid gender values")
else:
    log_test(table_name, "Validity", "cst_gndr valid values", "PASS", "")

print("✅ crm_customer tests completed")

## 2. bike_store.bronze.crm_product
Product data from CRM system

In [0]:
table_name = "bike_store.bronze.crm_product"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["prd_id", "prd_key", "prd_nm"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "prd_id": "int",
    "prd_key": "string",
    "prd_nm": "string",
    "prd_cost": "int",
    "prd_line": "string",
    "prd_start_dt": "date",
    "prd_end_dt": "date",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Range validations
check_range(df, table_name, "prd_cost", min_val=0)
check_range(df, table_name, "prd_start_dt", max_val=datetime.now().date())

# 4. Data quality checks
# Check for duplicate product IDs
dup_count = df.groupBy("prd_id").count().filter("count > 1").count()
if dup_count > 0:
    log_test(table_name, "Uniqueness", "prd_id unique", "FAIL", f"{dup_count} duplicate IDs")
else:
    log_test(table_name, "Uniqueness", "prd_id unique", "PASS", "")

# Check that end date is after start date
invalid_dates = df.filter(
    (F.col("prd_end_dt").isNotNull()) & 
    (F.col("prd_start_dt").isNotNull()) & 
    (F.col("prd_end_dt") < F.col("prd_start_dt"))
).count()
if invalid_dates > 0:
    log_test(table_name, "Logic", "prd_end_dt >= prd_start_dt", "FAIL", 
            f"{invalid_dates} records with end date before start date")
else:
    log_test(table_name, "Logic", "prd_end_dt >= prd_start_dt", "PASS", "")

# Known issue: prd_cost stored as INT (should be DECIMAL)
log_test(table_name, "Known Issues", "prd_cost type", "WARN", 
        "Cost stored as INT - should be DECIMAL in Silver layer")

print("✅ crm_product tests completed")

## 3. bike_store.bronze.crm_sales
Sales transaction data from CRM system

In [0]:
table_name = "bike_store.bronze.crm_sales"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["sls_ord_num", "sls_prd_key", "sls_cust_id", "sls_quantity", "sls_sales"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "sls_ord_num": "string",
    "sls_prd_key": "string",
    "sls_cust_id": "int",
    "sls_order_dt": "int",
    "sls_ship_dt": "int",
    "sls_due_dt": "int",
    "sls_sales": "int",
    "sls_quantity": "int",
    "sls_price": "int",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Range validations
check_range(df, table_name, "sls_sales", min_val=0)
check_range(df, table_name, "sls_quantity", min_val=0)
check_range(df, table_name, "sls_price", min_val=0)

# 4. Relationship validations
# Check FK to crm_customer
customer_df = spark.table("bike_store.bronze.crm_customer")
valid_customer_ids = customer_df.select("cst_id").distinct()
orphaned_sales = df.join(valid_customer_ids, df.sls_cust_id == valid_customer_ids.cst_id, "left_anti").count()
if orphaned_sales > 0:
    log_test(table_name, "Referential Integrity", "sls_cust_id FK valid", "FAIL", 
            f"{orphaned_sales} sales with invalid customer IDs")
else:
    log_test(table_name, "Referential Integrity", "sls_cust_id FK valid", "PASS", "")

# Check FK to crm_product
product_df = spark.table("bike_store.bronze.crm_product")
valid_product_keys = product_df.select("prd_key").distinct()
orphaned_sales_prd = df.join(valid_product_keys, df.sls_prd_key == valid_product_keys.prd_key, "left_anti").count()
if orphaned_sales_prd > 0:
    log_test(table_name, "Referential Integrity", "sls_prd_key FK valid", "FAIL", 
            f"{orphaned_sales_prd} sales with invalid product keys")
else:
    log_test(table_name, "Referential Integrity", "sls_prd_key FK valid", "PASS", "")

# 5. Data quality checks
# Check quantity > 0
zero_qty = df.filter(F.col("sls_quantity") <= 0).count()
if zero_qty > 0:
    log_test(table_name, "Logic", "sls_quantity > 0", "FAIL", f"{zero_qty} records with quantity <= 0")
else:
    log_test(table_name, "Logic", "sls_quantity > 0", "PASS", "")

# Known issues: dates stored as INT
log_test(table_name, "Known Issues", "Date columns type", "WARN", 
        "sls_order_dt, sls_ship_dt, sls_due_dt stored as INT - should be DATE in Silver layer")

# Known issue: monetary values as INT
log_test(table_name, "Known Issues", "Monetary columns type", "WARN", 
        "sls_sales, sls_price stored as INT - should be DECIMAL in Silver layer")

print("✅ crm_sales tests completed")

## 4. bike_store.bronze.erp_category
Category data from ERP system

In [0]:
table_name = "bike_store.bronze.erp_category"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["ID", "CAT"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "ID": "string",
    "CAT": "string",
    "SUBCAT": "string",
    "MAINTENANCE": "string",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Data quality checks
# Check for duplicate category IDs
dup_count = df.groupBy("ID").count().filter("count > 1").count()
if dup_count > 0:
    log_test(table_name, "Uniqueness", "ID unique", "FAIL", f"{dup_count} duplicate IDs")
else:
    log_test(table_name, "Uniqueness", "ID unique", "PASS", "")

# Check naming convention consistency
log_test(table_name, "Naming Convention", "Column names", "INFO", 
        "ERP tables use UPPERCASE - inconsistent with CRM (lowercase with underscores)")

print("✅ erp_category tests completed")

## 5. bike_store.bronze.erp_customer
Customer data from ERP system

In [0]:
table_name = "bike_store.bronze.erp_customer"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["CID"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "CID": "string",
    "BDATE": "date",
    "GEN": "string",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Range validations
# Birth date should be in the past and reasonable
from datetime import date
check_range(df, table_name, "BDATE", 
           min_val=date(1900, 1, 1), 
           max_val=date.today())

# 4. Data quality checks
# Check for duplicate customer IDs
dup_count = df.groupBy("CID").count().filter("count > 1").count()
if dup_count > 0:
    log_test(table_name, "Uniqueness", "CID unique", "FAIL", f"{dup_count} duplicate IDs")
else:
    log_test(table_name, "Uniqueness", "CID unique", "PASS", "")

# Check gender values
valid_genders = df.filter(F.col("GEN").isin(["M", "F", "Male", "Female"])).count()
total = df.filter(F.col("GEN").isNotNull()).count()
if valid_genders < total:
    log_test(table_name, "Validity", "GEN valid values", "WARN", 
            f"{total - valid_genders} invalid gender values")
else:
    log_test(table_name, "Validity", "GEN valid values", "PASS", "")

print("✅ erp_customer tests completed")

## 6. bike_store.bronze.erp_location
Location data from ERP system

In [0]:
table_name = "bike_store.bronze.erp_location"
df = spark.table(table_name)

print(f"Testing {table_name}...")
print(f"Total records: {df.count():,}\n")

# 1. Completeness checks
required_cols = ["CID", "CNTRY"]
check_completeness(df, table_name, required_cols)

# 2. Schema consistency
expected_schema = {
    "CID": "string",
    "CNTRY": "string",
    "file_name": "string",
    "ingest_datetime": "timestamp"
}
check_schema(df, table_name, expected_schema)

# 3. Relationship validations
# Check FK to erp_customer
erp_customer_df = spark.table("bike_store.bronze.erp_customer")
valid_customer_ids = erp_customer_df.select("CID").distinct()
orphaned_locations = df.join(valid_customer_ids, df.CID == valid_customer_ids.CID, "left_anti").count()
if orphaned_locations > 0:
    log_test(table_name, "Referential Integrity", "CID FK valid", "FAIL", 
            f"{orphaned_locations} locations with invalid customer IDs")
else:
    log_test(table_name, "Referential Integrity", "CID FK valid", "PASS", "")

# 4. Data quality checks
# Check country code format (should be 2-3 letters)
invalid_country = df.filter(
    (F.col("CNTRY").isNotNull()) & 
    ((F.length(F.col("CNTRY")) < 2) | (F.length(F.col("CNTRY")) > 3))
).count()
if invalid_country > 0:
    log_test(table_name, "Validity", "CNTRY format", "WARN", 
            f"{invalid_country} records with invalid country code length")
else:
    log_test(table_name, "Validity", "CNTRY format", "PASS", "")

print("✅ erp_location tests completed")

## Test Summary Report
Comprehensive overview of all data quality test results

In [0]:
# Convert test results to DataFrame
results_df = spark.createDataFrame(pd.DataFrame(test_results))

# Display full results
print("="*100)
print("DETAILED TEST RESULTS")
print("="*100)
display(results_df.orderBy("table", "category", "test"))

# Summary by status
print("\n" + "="*100)
print("SUMMARY BY STATUS")
print("="*100)
summary_status = results_df.groupBy("status").count().orderBy("status")
display(summary_status)

# Summary by table
print("\n" + "="*100)
print("SUMMARY BY TABLE")
print("="*100)
summary_table = results_df.groupBy("table", "status").count().orderBy("table", "status")
display(summary_table)

# Summary by category
print("\n" + "="*100)
print("SUMMARY BY CATEGORY")
print("="*100)
summary_category = results_df.groupBy("category", "status").count().orderBy("category", "status")
display(summary_category)

# Failed tests
failed_tests = results_df.filter(F.col("status") == "FAIL")
failed_count = failed_tests.count()

print("\n" + "="*100)
if failed_count > 0:
    print(f"⚠️  CRITICAL: {failed_count} TESTS FAILED")
    print("="*100)
    display(failed_tests.select("table", "category", "test", "details"))
else:
    print("✅ ALL TESTS PASSED (no failures detected)")
    print("="*100)

# Warnings
warn_tests = results_df.filter(F.col("status") == "WARN")
warn_count = warn_tests.count()

if warn_count > 0:
    print("\n" + "="*100)
    print(f"⚠️  {warn_count} WARNINGS DETECTED")
    print("="*100)
    display(warn_tests.select("table", "category", "test", "details"))

print("\n" + "="*100)
print("TEST EXECUTION COMPLETED")
print("="*100)
print(f"Total tests executed: {len(test_results)}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Export Critical Failures to CSV
Export failed tests to CSV file for reporting and tracking

In [0]:
# Filter for failed tests only
failed_tests_df = results_df.filter(F.col("status") == "FAIL")

# Select relevant columns and convert to Pandas for CSV export
failed_export = failed_tests_df.select(
    "table", 
    "category", 
    "test", 
    "details",
    "timestamp"
).toPandas()

# Define output path
output_path = "/Workspace/Users/adriano.study31@gmail.com/Project_bikestore/critical_failures_bronze.csv"

# Save to CSV
failed_export.to_csv(output_path, index=False)

print(f"✅ Critical failures exported successfully!")
print(f"📁 File location: {output_path}")
print(f"📊 Total failures exported: {len(failed_export)}")
print(f"\nPreview of exported data:")
display(failed_export)

## Análise Detalhada de Qualidade de Dados
Análise completa de colunas suspeitas, nulls, formatos, anomalias, duplicados e tipos errados

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

# Initialize analysis results
analysis_results = {}

def analyze_table(table_name, df):
    """Perform comprehensive analysis on a table"""
    print(f"\n{'='*100}")
    print(f"Analyzing: {table_name}")
    print(f"{'='*100}")
    
    total_rows = df.count()
    print(f"Total Records: {total_rows:,}\n")
    
    results = {
        'table': table_name,
        'total_rows': total_rows,
        'columns': [],
        'nulls': [],
        'duplicates': [],
        'type_issues': [],
        'anomalies': [],
        'suspicious_columns': []
    }
    
    # 1. ANALYZE EACH COLUMN
    print("\n📊 COLUMN ANALYSIS:")
    print("-" * 100)
    
    for col_name, col_type in df.dtypes:
        col_info = {
            'name': col_name,
            'type': col_type,
            'null_count': 0,
            'null_pct': 0,
            'distinct_count': 0,
            'min': None,
            'max': None,
            'issues': []
        }
        
        # Null analysis
        null_count = df.filter(F.col(col_name).isNull()).count()
        null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0
        col_info['null_count'] = null_count
        col_info['null_pct'] = null_pct
        
        # Distinct values
        distinct_count = df.select(col_name).distinct().count()
        col_info['distinct_count'] = distinct_count
        
        # Get min/max for numeric and date columns
        if col_type in ['int', 'bigint', 'double', 'float', 'decimal']:
            stats = df.agg(
                F.min(col_name).alias('min'),
                F.max(col_name).alias('max')
            ).collect()[0]
            col_info['min'] = stats['min']
            col_info['max'] = stats['max']
        elif col_type == 'date':
            stats = df.agg(
                F.min(col_name).alias('min'),
                F.max(col_name).alias('max')
            ).collect()[0]
            col_info['min'] = str(stats['min']) if stats['min'] else None
            col_info['max'] = str(stats['max']) if stats['max'] else None
        
        # Flag issues
        if null_pct > 5:
            col_info['issues'].append(f"High null rate: {null_pct:.1f}%")
            results['nulls'].append({
                'column': col_name,
                'null_count': null_count,
                'null_pct': null_pct
            })
        
        # Check if all values are the same (suspicious)
        if distinct_count == 1 and total_rows > 1:
            col_info['issues'].append("All values are identical")
            results['suspicious_columns'].append({
                'column': col_name,
                'reason': 'All values are identical'
            })
        
        # Check if column should be a different type
        if col_type == 'int' and col_name.lower().endswith('_dt'):
            col_info['issues'].append("DATE stored as INT")
            results['type_issues'].append({
                'column': col_name,
                'current_type': col_type,
                'suggested_type': 'DATE',
                'reason': 'Column name suggests date but stored as INT'
            })
        
        if col_type == 'int' and any(x in col_name.lower() for x in ['cost', 'price', 'sales', 'amount']):
            col_info['issues'].append("MONETARY value stored as INT")
            results['type_issues'].append({
                'column': col_name,
                'current_type': col_type,
                'suggested_type': 'DECIMAL',
                'reason': 'Monetary value should use DECIMAL for precision'
            })
        
        results['columns'].append(col_info)
    
    # 2. CHECK FOR DUPLICATE ROWS
    print("\n🔍 DUPLICATE ANALYSIS:")
    print("-" * 100)
    
    # Identify potential key columns (id, key, num)
    key_columns = [col for col, _ in df.dtypes if any(x in col.lower() for x in ['id', 'key', 'num', 'cid'])]
    
    for key_col in key_columns:
        dup_count = df.groupBy(key_col).count().filter("count > 1").count()
        total_distinct = df.select(key_col).distinct().count()
        
        if dup_count > 0:
            print(f"⚠️  {key_col}: {dup_count} duplicate values found")
            results['duplicates'].append({
                'column': key_col,
                'duplicate_count': dup_count,
                'distinct_values': total_distinct
            })
        else:
            print(f"✅ {key_col}: No duplicates (unique key)")
    
    # 3. DETECT ANOMALIES
    print("\n🚨 ANOMALY DETECTION:")
    print("-" * 100)
    
    for col_name, col_type in df.dtypes:
        if col_type in ['int', 'bigint', 'double', 'float']:
            # Check for negative values where they shouldn't exist
            if any(x in col_name.lower() for x in ['quantity', 'qty', 'sales', 'price', 'cost', 'amount']):
                negative_count = df.filter(F.col(col_name) < 0).count()
                if negative_count > 0:
                    print(f"⚠️  {col_name}: {negative_count} negative values (should be >= 0)")
                    results['anomalies'].append({
                        'column': col_name,
                        'anomaly': 'Negative values',
                        'count': negative_count
                    })
            
            # Check for zero values where they shouldn't exist
            if any(x in col_name.lower() for x in ['quantity', 'qty']):
                zero_count = df.filter(F.col(col_name) == 0).count()
                if zero_count > 0:
                    print(f"⚠️  {col_name}: {zero_count} zero values (questionable for quantity)")
                    results['anomalies'].append({
                        'column': col_name,
                        'anomaly': 'Zero values',
                        'count': zero_count
                    })
        
        elif col_type == 'date':
            # Check for future dates
            future_count = df.filter(F.col(col_name) > F.current_date()).count()
            if future_count > 0:
                print(f"⚠️  {col_name}: {future_count} future dates")
                results['anomalies'].append({
                    'column': col_name,
                    'anomaly': 'Future dates',
                    'count': future_count
                })
            
            # Check for very old dates (before 1900)
            old_count = df.filter(F.col(col_name) < F.lit('1900-01-01')).count()
            if old_count > 0:
                print(f"⚠️  {col_name}: {old_count} dates before 1900")
                results['anomalies'].append({
                    'column': col_name,
                    'anomaly': 'Dates before 1900',
                    'count': old_count
                })
    
    analysis_results[table_name] = results
    return results

print("\n" + "="*100)
print("COMPREHENSIVE DATA QUALITY ANALYSIS - BRONZE SCHEMA")
print("="*100)

In [0]:
# Analyze all 6 bronze tables
tables_to_analyze = [
    "bike_store.bronze.crm_customer",
    "bike_store.bronze.crm_product",
    "bike_store.bronze.crm_sales",
    "bike_store.bronze.erp_category",
    "bike_store.bronze.erp_customer",
    "bike_store.bronze.erp_location"
]

for table_name in tables_to_analyze:
    df = spark.table(table_name)
    analyze_table(table_name, df)

print("\n" + "="*100)
print("✅ ANALYSIS COMPLETED FOR ALL TABLES")
print("="*100)

In [0]:
# Generate comprehensive summary report
print("\n" + "="*100)
print("📋 CONSOLIDATED SUMMARY REPORT")
print("="*100)

# 1. NULL ISSUES SUMMARY
print("\n" + "="*100)
print("1️⃣  HIGH NULL RATE COLUMNS (>5%)")
print("="*100)

null_issues = []
for table_name, results in analysis_results.items():
    for null_info in results['nulls']:
        null_issues.append({
            'table': table_name.split('.')[-1],
            'column': null_info['column'],
            'null_count': null_info['null_count'],
            'null_pct': null_info['null_pct']
        })

if null_issues:
    null_df = pd.DataFrame(null_issues).sort_values('null_pct', ascending=False)
    display(null_df)
else:
    print("✅ No columns with high null rates")

# 2. TYPE ISSUES SUMMARY
print("\n" + "="*100)
print("2️⃣  TYPE MISMATCHES")
print("="*100)

type_issues = []
for table_name, results in analysis_results.items():
    for type_info in results['type_issues']:
        type_issues.append({
            'table': table_name.split('.')[-1],
            'column': type_info['column'],
            'current_type': type_info['current_type'],
            'suggested_type': type_info['suggested_type'],
            'reason': type_info['reason']
        })

if type_issues:
    type_df = pd.DataFrame(type_issues)
    display(type_df)
else:
    print("✅ No type mismatches detected")

# 3. DUPLICATE ISSUES
print("\n" + "="*100)
print("3️⃣  DUPLICATE KEY VALUES")
print("="*100)

dup_issues = []
for table_name, results in analysis_results.items():
    for dup_info in results['duplicates']:
        dup_issues.append({
            'table': table_name.split('.')[-1],
            'column': dup_info['column'],
            'duplicate_count': dup_info['duplicate_count'],
            'distinct_values': dup_info['distinct_values']
        })

if dup_issues:
    dup_df = pd.DataFrame(dup_issues)
    display(dup_df)
else:
    print("✅ No duplicate key values detected")

# 4. ANOMALIES
print("\n" + "="*100)
print("4️⃣  DATA ANOMALIES")
print("="*100)

anomaly_issues = []
for table_name, results in analysis_results.items():
    for anomaly_info in results['anomalies']:
        anomaly_issues.append({
            'table': table_name.split('.')[-1],
            'column': anomaly_info['column'],
            'anomaly_type': anomaly_info['anomaly'],
            'count': anomaly_info['count']
        })

if anomaly_issues:
    anomaly_df = pd.DataFrame(anomaly_issues)
    display(anomaly_df)
else:
    print("✅ No anomalies detected")

# 5. SUSPICIOUS COLUMNS
print("\n" + "="*100)
print("5️⃣  SUSPICIOUS COLUMNS")
print("="*100)

suspicious = []
for table_name, results in analysis_results.items():
    for susp_info in results['suspicious_columns']:
        suspicious.append({
            'table': table_name.split('.')[-1],
            'column': susp_info['column'],
            'reason': susp_info['reason']
        })

if suspicious:
    susp_df = pd.DataFrame(suspicious)
    display(susp_df)
else:
    print("✅ No suspicious columns detected")

print("\n" + "="*100)
print("📊 OVERALL SUMMARY")
print("="*100)
print(f"Total Tables Analyzed: {len(analysis_results)}")
print(f"High Null Rate Columns: {len(null_issues)}")
print(f"Type Mismatches: {len(type_issues)}")
print(f"Duplicate Issues: {len(dup_issues)}")
print(f"Anomalies Found: {len(anomaly_issues)}")
print(f"Suspicious Columns: {len(suspicious)}")
print("="*100)